# 🍽️ Google Maps Platform + Gemini 3.5 Flash AI 음식점 분석 & 맞춤 여행 코스 추천
### — Google Maps Web Service API 전수 활용 & Gemini Flash 기반 지능형 맛집·카페·여행 코스 생성 시스템

본 노트북은 **Google Maps Platform API**(Places New, Geocoding, Directions 등)와 **Google Gemini 3.5 Flash LLM**을 결합하여, 특정 음식점을 기준으로 아래의 5가지 실무 시나리오를 자동으로 분석하고 맞춤형 여행 코스를 도출합니다.

---

### 🗺️ 5단계 지능형 분석 파이프라인
```
┌────────────────────────────────────────────────────────────────────────┐
│ [사용자 입력: 음식점 검색어 (예: '명동교자 본점')]                      │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 1️⃣ 음식점 기본 정보 & 편의시설 카드 (Places API New Details `*`)       │
│    • 도로명 주소 & 우편번호, 영업시간/브레이크타임, 전화번호           │
│    • 유아의자, 화장실, 단체석, 주차, 예약가능, 포장, 배달              │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 2️⃣ 리뷰 기반 인기 메뉴 분석 (Places Reviews ➡️ Gemini 3.5 Flash)      │
│    • 실제 방문자 리뷰 원문/번역본 + 에디토리얼 요약 수집               │
│    • Gemini AI가 대표 시그니처 메뉴, 맛의 특징, 추천 조합 분석         │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 3️⃣ 비슷한 맛집 추천 (Places SearchNearby ➡️ Gemini 3.5 Flash)         │
│    • 동일 카테고리/가격대 반경 2km 내 맛집 탐색                         │
│    • Gemini AI가 메뉴/분위기 비교 및 추천 이유 요약                    │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 4️⃣ 식사 후 추천 근처 카페 (Places SearchNearby ➡️ Gemini 3.5 Flash)   │
│    • 도보 5~10분(500m~800m) 내 평점 4.0+ 카페/베이커리 탐색            │
│    • 실시간 도보 거리 및 식후 입가심 디저트/커피 큐레이션              │
└───────────────────────────────────┬────────────────────────────────────┘
                                    │
                                    ▼
┌────────────────────────────────────────────────────────────────────────┐
│ 5️⃣ 맞춤형 여행 코스 생성 (Places + Directions API + Gemini Flash)     │
│    🚗 차량 여행 코스: 15km 내 드라이브 뷰포인트, 주차 여부, 소요시간   │
│    🚶 도보 여행 코스: 20분 내 힐링 산책로/문화거리, 턴바이턴 보행 가이드 │
└────────────────────────────────────────────────────────────────────────┘
```

---


## 📦 0. 환경 설정 및 API 키 로드

`.env` 파일에 설정된 `GOOGLE_MAPS_API_KEY`와 `GEMINI_API_KEY`를 자동으로 로드하고 클라이언트를 초기화합니다.


In [1]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
import googlemaps
from google import genai
from google.genai import types

# .env 자동 탐색 및 로드
load_dotenv(find_dotenv(), override=True)

MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY", "").strip().strip('"').strip("'")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip().strip('"').strip("'")

# 1. Google Maps SDK 초기화
if not MAPS_API_KEY:
    import getpass
    MAPS_API_KEY = getpass.getpass("GOOGLE_MAPS_API_KEY를 입력하세요: ").strip()

gmaps = googlemaps.Client(key=MAPS_API_KEY)
masked_maps_key = f"{MAPS_API_KEY[:6]}...{MAPS_API_KEY[-4:]}" if len(MAPS_API_KEY) > 10 else "***"
print(f"✅ Google Maps 클라이언트 초기화 완료 (키: {masked_maps_key})")

# 2. Gemini 3.5 Flash 클라이언트 초기화
if not GEMINI_API_KEY:
    import getpass
    GEMINI_API_KEY = getpass.getpass("GEMINI_API_KEY를 입력하세요: ").strip()

ai_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.5-flash"
print(f"✅ Gemini 클라이언트 초기화 완료 (모델: {GEMINI_MODEL})")


✅ Google Maps 클라이언트 초기화 완료 (키: AIzaSy...wVj0)
✅ Gemini 클라이언트 초기화 완료 (모델: gemini-3.5-flash)


## 🎯 분석 대상 음식점 설정
원하는 음식점 이름을 아래 변수에 입력하면 전체 분석 및 코스가 자동으로 생성됩니다.


In [2]:
# 분석하고자 하는 음식점 상호명 또는 주소
TARGET_RESTAURANT = "명동교자 본점"
print(f"🎯 분석 대상 음식점: '{TARGET_RESTAURANT}'")


🎯 분석 대상 음식점: '명동교자 본점'


## 📋 1. 음식점 기본 정보 & 편의시설 조회
- **주소**: 도로명 주소, 우편번호, 좌표
- **영업시간 & 브레이크타임**: 요일별 운영 시간 및 쉬는 시간
- **전화번호**: 대표 연락처
- **부대시설**: 유아의자, 화장실, 단체석, 주차, 예약가능, 포장, 배달, 결제수단


In [3]:
# 1.1 Places API Text Search: 음식점 검색 및 Place ID 추출
search_url = "https://places.googleapis.com/v1/places:searchText"
search_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.primaryType"
}
search_body = {
    "textQuery": TARGET_RESTAURANT,
    "languageCode": "ko",
    "regionCode": "kr"
}

search_res = requests.post(search_url, headers=search_headers, json=search_body).json()
places_found = search_res.get("places", [])

if not places_found:
    raise ValueError(f"'{TARGET_RESTAURANT}'을(를) 찾을 수 없습니다. 검색어를 확인해주세요.")

target_place = places_found[0]
target_place_id = target_place["id"]
target_lat = target_place["location"]["latitude"]
target_lng = target_place["location"]["longitude"]
target_coords = (target_lat, target_lng)

print(f"✅ 음식점 발견: {target_place.get('displayName', {}).get('text')} (Place ID: {target_place_id})")
print(f"📍 좌표: lat={target_lat}, lng={target_lng}")

# 1.2 Places API Details: 와일드카드 '*'로 50+ 전체 상세 속성 조회
details_url = f"https://places.googleapis.com/v1/places/{target_place_id}"
details_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "*"
}
details_data = requests.get(details_url, headers=details_headers, params={"languageCode": "ko"}).json()

# 1.3 기본 정보 구조화
basic_info = {
    "상호명": details_data.get("displayName", {}).get("text"),
    "대표 카테고리": details_data.get("primaryType", "음식점"),
    "표준 도로명 주소": details_data.get("formattedAddress"),
    "전화번호 (국번)": details_data.get("nationalPhoneNumber", "제공안됨"),
    "국제 전화번호": details_data.get("internationalPhoneNumber", "제공안됨"),
    "웹사이트": details_data.get("websiteUri", "없음"),
    "Google 지도 링크": details_data.get("googleMapsUri"),
    "전체 평점": f"⭐ {details_data.get('rating', 'N/A')} / 5.0 (리뷰 {details_data.get('userRatingCount', 0):,}개)"
}

print("📌 [1. 기본 정보 요약]")
for k, v in basic_info.items():
    print(f"  • {k}: {v}")

# 1.4 영업시간 및 브레이크타임
opening_hours = details_data.get("regularOpeningHours", {})
weekday_descriptions = opening_hours.get("weekdayDescriptions", [])
print("\n⏰ [2. 요일별 영업시간 & 브레이크타임]")
if weekday_descriptions:
    for desc in weekday_descriptions:
        print(f"  • {desc}")
else:
    print("  • 영업시간 정보가 등록되어 있지 않습니다.")

# 1.5 편의 및 부대시설 테이블
amenities = {
    "유아의자 / 어린이 메뉴": "✅ 제공" if details_data.get("menuForChildren") or details_data.get("goodForChildren") else ("❌ 미제공" if details_data.get("goodForChildren") is False else "정보없음"),
    "화장실 구비": "✅ 구비" if details_data.get("restroom") else "정보없음",
    "휠체어 접근 가능 화장실": "✅ 가능" if details_data.get("accessibilityOptions", {}).get("wheelchairAccessibleRestroom") else "확인필요",
    "휠체어 출입구": "✅ 완비" if details_data.get("accessibilityOptions", {}).get("wheelchairAccessibleEntrance") else "확인필요",
    "단체 이용 가능 (단체의석)": "✅ 가능" if details_data.get("goodForGroups") else ("❌ 불가" if details_data.get("goodForGroups") is False else "정보없음"),
    "야외 좌석 (테라스)": "✅ 완비" if details_data.get("outdoorSeating") else ("❌ 없음" if details_data.get("outdoorSeating") is False else "정보없음"),
    "주차 시설": "🅿️ 유료/무료 주차 제공" if any(details_data.get("parkingOptions", {}).values()) else "❌ 전용 주차장 없음 (인근 유료주차 권장)",
    "예약 가능 여부": "✅ 예약 가능" if details_data.get("reservable") else ("❌ 현장 대기" if details_data.get("reservable") is False else "확인필요"),
    "포장 (Takeout)": "✅ 가능" if details_data.get("takeout") else "정보없음",
    "배달 (Delivery)": "✅ 가능" if details_data.get("delivery") else "정보없음",
    "반려동물 동반": "🐾 가능" if details_data.get("allowsDogs") else ("❌ 불가" if details_data.get("allowsDogs") is False else "정보없음")
}

df_amenities = pd.DataFrame(list(amenities.items()), columns=["시설 및 서비스 항목", "제공 여부"])
display(df_amenities)


✅ 음식점 발견: 명동교자 본점 (Place ID: ChIJW7FBDfCifDURHTpisLbVUH0)
📍 좌표: lat=37.5610151, lng=126.9860829
📌 [1. 기본 정보 요약]
  • 상호명: 명동교자 본점
  • 대표 카테고리: dumpling_restaurant
  • 표준 도로명 주소: 대한민국 서울특별시 중구 퇴계로 129
  • 전화번호 (국번): 02-776-5348
  • 국제 전화번호: +82 2-776-5348
  • 웹사이트: http://www.mdkj.co.kr/
  • Google 지도 링크: https://maps.google.com/?cid=9029952233497836061&g_mp=CiVnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLkdldFBsYWNlEAIYBCAA
  • 전체 평점: ⭐ 4.2 / 5.0 (리뷰 14,724개)

⏰ [2. 요일별 영업시간 & 브레이크타임]
  • 월요일: 오전 10:30 ~ 오후 9:00
  • 화요일: 오전 10:30 ~ 오후 9:00
  • 수요일: 오전 10:30 ~ 오후 9:00
  • 목요일: 오전 10:30 ~ 오후 9:00
  • 금요일: 오전 10:30 ~ 오후 9:00
  • 토요일: 오전 10:30 ~ 오후 9:00
  • 일요일: 오전 10:30 ~ 오후 9:00


,시설 및 서비스 항목,제공 여부
0,유아의자 / 어린이 메뉴,정보없음
1,화장실 구비,✅ 구비
2,휠체어 접근 가능 화장실,확인필요
3,휠체어 출입구,✅ 완비
4,단체 이용 가능 (단체의석),✅ 가능
5,야외 좌석 (테라스),❌ 없음
6,주차 시설,❌ 전용 주차장 없음 (인근 유료주차 권장)
7,예약 가능 여부,❌ 현장 대기
8,포장 (Takeout),✅ 가능
9,배달 (Delivery),정보없음


## 🍜 2. 메뉴 정보 (구글 맵 리뷰 기반 인기 메뉴 분석)
Places API에서 수집된 **실제 방문자 리뷰(원문/한국어 번역)**와 **에디토리얼 요약(`editorialSummary`)**을 **Gemini 3.5 Flash** 모델에 전달하여 고객들이 가장 많이 찾는 대표 인기 메뉴와 맛의 특징을 분석합니다.


In [4]:
# 2.1 리뷰 데이터 및 에디토리얼 요약 수집
reviews_data = details_data.get("reviews", [])
editorial_summary = details_data.get("editorialSummary", {}).get("text", "에디토리얼 요약 정보 없음")

reviews_text_list = []
for idx, r in enumerate(reviews_data):
    author = r.get("authorAttribution", {}).get("displayName", "익명")
    rating = r.get("rating", 5)
    text = r.get("text", {}).get("text", "")
    orig_text = r.get("originalText", {}).get("text", text)
    rel_time = r.get("relativePublishTimeDescription", "")
    reviews_text_list.append(f"[리뷰 {idx+1}] 작성자: {author} (⭐{rating}점, {rel_time})\n- 내용: {text}\n- 원문: {orig_text}")

combined_reviews_text = "\n\n".join(reviews_text_list)

# 2.2 Gemini 3.5 Flash 프롬프트 작성
menu_prompt = f"""
당신은 대한민국 최고의 미식 전문 AI 큐레이터입니다.
아래는 Google Maps Platform에서 수집한 '{TARGET_RESTAURANT}'의 실제 고객 리뷰 및 소개 요약 데이터입니다.

[Google 에디토리얼 요약]
{editorial_summary}

[실제 고객 방문 리뷰]
{combined_reviews_text}

위 데이터를 정밀 분석하여 다음 내용을 마크다운으로 명확하게 정리해주세요:
1. 🔥 **사람들이 가장 즐겨 찾는 대표 인기 메뉴 Top 3~4** (메뉴명, 특징, 고객들의 추천 이유)
2. 👅 **맛과 식감의 핵심 포인트** (육수의 풍미, 면발, 곁들임 김치/반찬의 조화, 양과 맵기 등)
3. 💡 **첫 방문자를 위한 꿀조합 및 팁** (선불/주문 방식, 사리/밥 추가, 방문 팁 등)
"""

print("🤖 Gemini 3.5 Flash 모델로 고객 리뷰 분석 중...\n")
response_menu = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=menu_prompt
)

from IPython.display import Markdown
display(Markdown(response_menu.text))


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


🤖 Gemini 3.5 Flash 모델로 고객 리뷰 분석 중...



안녕하세요! 대한민국 최고의 미식 전문 AI 큐레이터입니다. 

오늘 소개해 드릴 곳은 오랜 세월 동안 명동을 지켜온 명불허전의 맛집, **‘명동교자 본점’**입니다. 수많은 국내외 미식가들의 실제 방문 리뷰를 바탕으로, 이곳의 매력과 이용 팁을 날카롭고 깊이 있게 분석해 드립니다.

---

### 1. 🔥 사람들이 가장 즐겨 찾는 대표 인기 메뉴 Top 3

명동교자는 단출한 메뉴 구성으로 전문성을 극대화한 곳입니다. 방문객들이 극찬하는 대표 메뉴 3가지를 소개합니다.

*   **① 명동칼국수 (signature)**
    *   **특징:** 일반적인 멸치나 바지락 칼국수와 달리, **진한 닭 육수**를 베이스로 합니다. 고명으로 볶은 양파와 간 고기가 올라가 고소하면서도 은은한 **불 맛(훈연향)**이 감도는 깊은 갈색빛 육수가 특징입니다. 얇은 피의 미니 만두(변시만두)가 고명으로 함께 제공됩니다.
    *   **추천 이유:** 깊고 묵직한 국물이 속을 제대로 풀어주며, '추억의 맛'을 찾는 이들에게 압도적인 지지를 받습니다. 갈비탕이나 중화풍 면요리를 연상시키는 독특하고 이국적인 풍미가 일품입니다.
*   **② 만두**
    *   **특징:** 얇고 투명한 만두피 속에 **부추와 돼지고기가 빈틈없이 꽉 차 있는** 정통 고기만두입니다. 한 판에 풍성하게 제공되어 비주얼부터 식욕을 자극합니다.
    *   **추천 이유:** 한 입 베어 물면 풍부한 육즙이 터져 나옵니다. 그냥 간장에 찍어 먹어도 훌륭하지만, 칼국수 국물에 담가 적셔 먹으면 한층 더 깊은 풍미를 즐길 수 있어 테이블마다 필수로 주문하는 메뉴입니다.
*   **③ 비빔국수**
    *   **특징:** **클로렐라를 넣어 만든 초록색 면발**이 시각적인 즐거움과 신선함을 선사합니다. 매콤하고 자극적인 양념장으로 맛을 냈습니다.
    *   **추천 이유:** 칼국수와 만두의 기름진 맛을 매콤하게 잡아주는 훌륭한 파트너입니다. 다만, 양념에 마늘 향이 강하게 배어 있어 알싸한 매운맛을 좋아하는 분들께 추천합니다.

---

### 2. 👅 맛과 식감의 핵심 포인트

명동교자의 음식들은 대중적이면서도 이곳만의 아주 강렬한 개성을 지니고 있습니다.

*   **육수의 풍미: 깊고 묵직한 감칠맛**
    *   닭 육수 특유의 진하고 묵직한 감칠맛이 특징입니다. 볶은 양파에서 우러난 단맛과 고기 고명이 어우러져 "숯불 소갈비를 구워 맛을 낸 듯한 깊은 맛"이라는 평가가 있을 정도로 묵직하고 고소합니다. 다만 깔끔하고 시원한 한국식 해물 칼국수 스타일에 익숙하다면 다소 기름지거나 중화풍 면요리처럼 느껴질 수 있습니다.
*   **면발: 부드럽게 넘어가는 극상의 부드러움**
    *   쫄깃하고 탱글한 식감보다는 **부드럽고 하늘하늘한 식감**입니다. 후루룩 쉽게 넘어가는 목 넘김이 특징이며, 푹 익은 면(퍼진 면)을 선호하는 분들에게 최고의 만족감을 줍니다.
*   **곁들임 마늘김치: 명동교자의 정체성**
    *   이곳의 김치는 **알싸하고 강렬한 마늘 양념**이 폭탄처럼 들어간 것으로 유명합니다. 혀가 아릴 정도로 매콤하고 알싸하지만, 자칫 느끼할 수 있는 닭 육수 칼국수와 곁들였을 때 환상의 궁합을 자랑합니다. 호불호가 극명히 갈리지만 한 번 빠지면 헤어날 수 없는 중독성을 자랑합니다.
*   **양과 서비스**
    *   기본적으로 만두와 국수 모두 양이 푸짐합니다. 이에 더해 인심 좋게 **면 사리 추가와 공깃밥이 무료로 제공**되므로 대식가들도 든든하게 식사를 마칠 수 있습니다.

---

### 3. 💡 첫 방문자를 위한 꿀조합 및 이용 팁

명동교자는 기업형으로 매우 빠르고 효율적으로 운영되는 매장입니다. 첫 방문 시 당황하지 않도록 아래 팁을 꼭 확인하세요.

*   **초고속 선불 시스템**
    *   매장에 입장하면서 카운터에서 **주문과 동시에 결제(선불)**를 진행합니다. 자리에 앉자마자 음식이 거의 즉시 서빙되므로, 웨이팅이 있더라도 회전율이 엄청나게 빨라 대기 시간(평균 5분 내외)이 길지 않습니다.
*   **미식 큐레이터 추천 꿀조합**
    *   **2인 방문 시:** `칼국수 1개 + 비빔국수 1개 + 만두 1판` 조합을 강력 추천합니다. 
    *   **먹는 순서:** 
        1. 먼저 육즙 가득한 만두를 간장에 찍어 즐깁니다.
        2. 칼국수 면을 강렬한 마늘김치 싸서 한 입 먹은 뒤, 진한 국물을 들이켭니다.
        3. 남은 만두를 칼국수 국물에 으깨어 만둣국처럼 즐깁니다.
        4. 마지막으로 서비스로 제공되는 **공깃밥(무료)**을 요청해 남은 칼국수 국물에 말아 '마늘김치'를 올려 마무리합니다.
*   **비방문자 주의 사항 및 매너 팁**
    *   **마늘 향 주의:** 김치와 비빔국수에 마늘이 상상 이상으로 많이 들어가므로, 식사 후 마늘 향이 강하게 남을 수 있습니다. (중요한 미팅이나 데이트 직전이라면 주의가 필요합니다!)
    *   **위생 부분:** 워낙 회전율이 빠르고 붐비는 매장이다 보니 식기류의 세척 상태에 예민한 분들은 호불호가 있을 수 있습니다.

## 🍲 3. 이 음식점과 비슷한 유사 맛집 추천
음식점의 카테고리(`primaryType`), 평점, 가격대, 지리적 위치를 기반으로 반경 2km 이내 유사 맛집을 탐색한 뒤, **Gemini 3.5 Flash**가 맛과 분위기를 대조하여 추천 사유를 제시합니다.


In [5]:
# 3.1 반경 2km 이내 동일/유사 카테고리 음식점 검색
nearby_res_url = "https://places.googleapis.com/v1/places:searchNearby"
nearby_res_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.primaryType,places.location"
}
nearby_res_body = {
    "includedTypes": ["korean_restaurant", "restaurant"],
    "maxResultCount": 6,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 2000.0  # 반경 2km
        }
    },
    "languageCode": "ko"
}

similar_raw = requests.post(nearby_res_url, headers=nearby_res_headers, json=nearby_res_body).json()
similar_places = [p for p in similar_raw.get("places", []) if p.get("id") != target_place_id][:4]

similar_candidates = []
for p in similar_places:
    similar_candidates.append({
        "상호명": p.get("displayName", {}).get("text"),
        "평점": f"⭐ {p.get('rating', 'N/A')}",
        "리뷰수": p.get("userRatingCount", 0),
        "주소": p.get("formattedAddress"),
        "Place ID": p.get("id")
    })

df_similar = pd.DataFrame(similar_candidates)
print(f"✅ 반경 2km 내 유사 맛집 후보 {len(similar_candidates)}곳 발견:")
display(df_similar)

# 3.2 Gemini 3.5 Flash에게 유사 맛집 비교 추천 요청
similar_prompt = f"""
기준 맛집: '{TARGET_RESTAURANT}' (상호명: {details_data.get('displayName', {}).get('text')})
아래는 Google Maps Platform에서 검색된 인근 맛집 후보 목록입니다:
{json.dumps(similar_candidates, ensure_ascii=False, indent=2)}

위 후보들 중 '{TARGET_RESTAURANT}'을 방문하려던 미식가가 함께 고려해볼 만한 유사/대체 맛집 2~3곳을 선정하고:
1. 각 식당의 매력 및 메뉴/분위기 비교 포인트
2. 본점 대신 또는 2차 식사 장소로 방문했을 때의 장점 (웨이팅 분산, 특색 있는 요리 등)
을 친절하게 비교 추천해주세요.
"""

response_similar = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=similar_prompt
)
display(Markdown(response_similar.text))


✅ 반경 2km 내 유사 맛집 후보 4곳 발견:


,상호명,평점,리뷰수,주소,Place ID
0,보코서울명동,⭐ 4.2,564,대한민국 서울특별시 중구 퇴계로 52,ChIJwdSjpvWifDURKT_mblhMHIA
1,오다리집 간장게장,⭐ 4.8,7025,"2F, 28 명동8나길 중구 서울특별시 대한민국",ChIJNbhaHPGifDURDCGKakYKwXw
2,부촌육회 본점,⭐ 4.4,2215,대한민국 서울특별시 종로구 종로 200-12,ChIJQR1ODmCjfDUREpgR7iCRQDI
3,BHC치킨 명동본점,⭐ 3.8,2263,대한민국 서울특별시 중구 명동7길 21,ChIJCT9h1O-ifDURQuD7FuVo3-Q


대한민국의 대표적인 맛집이자 미쉐린 가이드 빕 구르망에 빛나는 **'명동교자 본점'**은 진한 닭 육수의 칼국수, 얇은 피의 만두, 그리고 알싸한 마늘 김치로 독보적인 사랑을 받는 곳입니다. 

이 명동교자를 방문하려던 미식가분들을 위해, 제시된 후보군 중 **대체 방문 및 2차 식사 장소로 가장 매력적인 3곳**을 엄선하여 비교 추천해 드립니다.

---

### 1. 오다리집 간장게장 (명동 내 최강의 감칠맛 대체지)
> **"명동교자만큼 강렬하고 중독성 있는 한국의 맛을 찾으신다면"**

*   **비교 포인트 (메뉴 및 분위기):**
    *   **명동교자**가 따뜻하고 든든한 탄수화물과 고기 육즙의 조화라면, **오다리집**은 한국 전통 ‘밥도둑’인 간장게장을 메인으로 하는 감칠맛의 끝판왕입니다. 
    *   외국인 관광객들에게도 인기가 많아 활기찬 분위기이며, 한식의 깊은 염도와 감칠맛을 즐길 수 있다는 점에서 명동교자의 '마늘 김치'가 주는 강렬한 자극을 대체하기에 충분합니다.
*   **방문 메리트 (대체 및 2차 장소 활용법):**
    *   **웨이팅 분산 및 메뉴 다변화:** 명동교자의 대기 줄이 지나치게 길 때 도보 거리 내에서 선택할 수 있는 훌륭한 대안입니다. 
    *   명동교자에서 면 요리로 가볍고 빠르게 식사를 마친 후, 조금 더 고급스럽고 술 한잔 곁들일 수 있는 식사를 원할 때 방문하기 좋습니다.

---

### 2. 부촌육회 본점 (미쉐린 빕 구르망의 품격 비교)
> **"명동교자와 같은 '미쉐린 빕 구르망' 타이틀의 깊이를 느끼고 싶다면"**

*   **비교 포인트 (메뉴 및 분위기):**
    *   두 곳 모두 **미쉐린 빕 구르망**에 다년간 선정된 검증된 맛집입니다. **명동교자**가 따뜻하게 끓여낸 정성의 맛이라면, **부촌육회**는 신선한 한우 우둔살의 쫄깃함과 고소한 참기름, 아삭한 배의 조화가 일품인 '날것의 미학'을 보여줍니다.
    *   활기찬 광장시장 골목에 위치하여 노포 특유의 정겨운 분위기를 느낄 수 있습니다.
*   **방문 메리트 (대체 및 2차 장소 활용법):**
    *   **미식 투어 코스 구성:** 명동교자에서 든든하게 칼국수로 배를 채운 뒤, 대중교통이나 택시로 가깝게 이동하여(종로5가 광장시장) 2차로 육회에 소주 한잔을 기울이기에 최고의 코스입니다. 
    *   명동의 현대적인 분위기에서 벗어나 서울의 전통 시장 감성을 느끼며 깔끔하고 단백질 위주의 안주로 하루를 마무리하고 싶을 때 강력 추천합니다.

---

### 3. BHC치킨 명동본점 (마늘 김치 뒤풀이를 위한 최고의 2차)
> **"명동교자의 강렬한 마늘 맛을 시원하게 씻어내 줄 '치맥' 한잔"**

*   **비교 포인트 (메뉴 및 분위기):**
    *   **명동교자**가 빠른 회전율 속에서 식사에 집중하는 분위기라면, **BHC치킨**은 넓고 캐주얼한 공간에서 동행인과 편안하게 대화를 나누며 즐길 수 있는 곳입니다.
    *   바삭한 튀김옷의 치킨(뿌링클, 맛초킹 등)과 시원한 맥주의 조합은 칼국수의 탄수화물과는 또 다른 즐거움을 줍니다.
*   **방문 메리트 (대체 및 2차 장소 활용법):**
    *   **완벽한 2차 장소:** 명동교자의 시그니처인 '알싸한 마늘 김치'를 먹고 나면 입안에 강한 여운이 남습니다. 이 입안을 시원한 생맥주와 바삭한 치킨으로 개운하게(?) 달래주는 코스로 제격입니다.
    *   명동 한복판에 위치하여 이동 동선이 매우 훌륭하며, 늦은 시간까지 운영하므로 명동교자의 이른 마감 시간(보통 21:00 전후) 이후 아쉬움을 달래기에 좋습니다.

---

### 요약 가이드
*   **명동을 벗어나지 않고 든든한 정식을 먹고 싶다면?** ➡️ **오다리집 간장게장**
*   **명동교자에 버금가는 또 다른 미쉐린 맛집을 경험하고 싶다면?** ➡️ **부촌육회 본점** (종로 이동)
*   **명동교자 식사 후, 편안하게 대화하며 맥주 한잔할 곳을 찾는다면?** ➡️ **BHC치킨 명동본점**

## ☕ 4. 식사 후 추천 근처 카페
음식점에서 식사를 마친 후 **도보 5~10분 (반경 500m~800m)** 내에 이동할 수 있는 평점 4.0 이상의 카페 및 베이커리를 검색하고, 식후 입가심에 어울리는 최적의 카페를 추천합니다.


In [6]:
# 4.1 도보 권역(반경 700m) 내 카페 검색
cafe_url = "https://places.googleapis.com/v1/places:searchNearby"
cafe_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.userRatingCount,places.location,places.outdoorSeating"
}
cafe_body = {
    "includedTypes": ["cafe", "coffee_shop", "bakery"],
    "maxResultCount": 5,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 700.0  # 도보 약 10분 이내
        }
    },
    "languageCode": "ko"
}

cafe_raw = requests.post(cafe_url, headers=cafe_headers, json=cafe_body).json()
cafe_places = cafe_raw.get("places", [])

# 4.2 직선거리 및 도보 시간 계산
import math
def haversine_meters(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

cafe_list = []
for c in cafe_places:
    c_lat = c["location"]["latitude"]
    c_lng = c["location"]["longitude"]
    dist_m = haversine_meters(target_lat, target_lng, c_lat, c_lng)
    walk_min = max(1, round(dist_m / 65))  # 평균 보행속도 분당 65m
    
    cafe_list.append({
        "카페명": c.get("displayName", {}).get("text"),
        "평점": f"⭐ {c.get('rating', 'N/A')}",
        "리뷰수": c.get("userRatingCount", 0),
        "도보 거리": f"{int(dist_m)} m",
        "도보 예상 시간": f"약 {walk_min} 분",
        "야외 좌석": "✅ 있음" if c.get("outdoorSeating") else "실내 좌석",
        "주소": c.get("formattedAddress"),
        "lat": c_lat,
        "lng": c_lng
    })

df_cafes = pd.DataFrame(cafe_list)
print(f"✅ 식후 도보 이동 가능한 근처 카페 {len(cafe_list)}곳 탐색 완료:")
display(df_cafes[["카페명", "평점", "리뷰수", "도보 거리", "도보 예상 시간", "야외 좌석", "주소"]])

# 4.3 Gemini 3.5 Flash의 식후 맞춤 카페 페어링 큐레이션
cafe_prompt = f"""
식사한 음식점: '{TARGET_RESTAURANT}' (진하고 깊은 국물/마늘 김치가 특징인 음식)
식후 방문 가능한 인근 카페 목록:
{json.dumps(cafe_list, ensure_ascii=False, indent=2)}

식사를 마친 손님이 입안을 깔끔하게 정리하고 담소를 나누기에 가장 적합한 카페 2~3곳을 선정하여:
1. 식후 음료/디저트 페어링 포인트 (예: 깔끔한 산미의 드립커피, 시원한 아메리카노, 시그니처 디저트)
2. 매장 분위기 및 좌석 편의성 (조용한 대화, 채광, 테라스 등)
을 추천해주세요.
"""

response_cafe = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=cafe_prompt
)
display(Markdown(response_cafe.text))


✅ 식후 도보 이동 가능한 근처 카페 5곳 탐색 완료:


,카페명,평점,리뷰수,도보 거리,도보 예상 시간,야외 좌석,주소
0,Rewire coffee,⭐ 4.9,526,536 m,약 8 분,✅ 있음,대한민국 서울특별시 중구 수표로6길 24-1
1,블루보틀 명동 카페,⭐ 4.5,255,395 m,약 6 분,실내 좌석,대한민국 서울특별시 중구 명동길 14
2,맷차 명동본점,⭐ 4.5,845,420 m,약 6 분,실내 좌석,대한민국 서울특별시 중구 명동9길 17
3,명동맘하우스,⭐ 4,259,81 m,약 1 분,실내 좌석,대한민국 서울특별시 중구 남산동3가 13-21
4,커피한약방,⭐ 4.5,2388,651 m,약 10 분,✅ 있음,대한민국 서울특별시 중구 삼일대로12길 16-6


진하고 깊은 고기 국물과 알싸하고 강렬한 마늘 김치가 특징인 '명동교자'에서 식사를 마치셨다면, **입안의 텁텁함과 마늘 향을 깔끔하게 잡아주고 편안하게 대화를 나눌 수 있는 카페**를 선택하시는 것이 좋습니다.

제시해주신 목록 중 이러한 목적(구취 제거, 입안 리프레시, 편안한 공간)에 가장 적합한 **카페 3곳**을 추천해 드립니다.

---

### 1. 맷차 명동본점 (도보 약 6분, 420m)
**"마늘 향을 완벽하게 잡아주는 진한 말차의 힘"**

*   **식후 음료/디저트 페어링 포인트:**
    *   **추천 음료:** **맷돌 말차 오리지널(또는 말차 밀크티)**
    *   **페어링 이유:** 녹차/말차에 풍부하게 함유된 '카테킨' 성분은 **마늘의 알리신 성분(냄새 유발 물질)을 분해하고 입안을 개운하게 만드는 데 탁월한 효과**가 있습니다. 맷돌로 직접 갈아 만든 맷차의 말차 음료는 인위적인 단맛 없이 쌉싸름하고 깔끔하여 명동교자의 묵직한 칼국수 국물 맛을 단번에 씻어내 줍니다.
*   **매장 분위기 및 좌석 편의성:**
    *   총 4층 규모의 대형 매장으로 공간이 매우 넓고 쾌적합니다. 명동 중심가에서 보기 드물게 **좌석 간격이 넓고 테이블이 커서** 식사 후 지인들과 눈치 보지 않고 여유롭게 장시간 담소를 나누기에 가장 좋은 환경을 제공합니다.

---

### 2. 커피한약방 (도보 약 10분, 651m)
**"산뜻한 필터 커피와 레트로 감성 속에서의 티타임"**

*   **식후 음료/디저트 페어링 포인트:**
    *   **추천 음료:** **필터 커피 (드립 커피 - 산미 있는 원두 추천)**
    *   **페어링 이유:** 에스프레소 머신으로 내린 커피보다 **종이 필터로 걸러낸 드립 커피가 오일감이 없어 입안을 가장 깔끔하게 정리**해 줍니다. 특히 과일 향과 산미가 있는 원두를 선택하시면 마늘 김치의 매운맛과 칼국수의 기름진 맛이 싹 사라지는 리프레시 효과를 느낄 수 있습니다. 맞은편 '혜민당'의 달콤한 타르트나 양과자를 곁들이면 완벽한 단짠 조화를 이룹니다.
*   **매장 분위기 및 좌석 편의성:**
    *   개화기 풍의 독특하고 아날로그한 인테리어가 돋보이는 곳입니다. 골목길 안쪽에 숨겨진 아지트 같은 느낌을 주며, **야외 좌석(테라스)이 마련되어 있어** 날씨가 좋을 때 바람을 쐬며 대화하기 좋습니다. 도보로 약 10분 정도 걸으며 자연스럽게 소화를 시킬 수 있다는 것도 장점입니다.

---

### 3. 블루보틀 명동 카페 (도보 약 6분, 395m)
**"정갈하고 미니멀한 공간에서 즐기는 스페셜티 커피"**

*   **식후 음료/디저트 페어링 포인트:**
    *   **추천 음료:** **싱글 오리진 푸어오버(Drip) 또는 놀라(Nola)**
    *   **페어링 이유:** 블루보틀 특유의 정교하게 추출된 푸어오버 커피는 깔끔함의 극치를 보여줍니다. 명동교자의 복합적이고 강한 양념 맛 뒤에 마시는 고품격 스페셜티 커피 한 잔은 미각을 차분하게 정돈해 줍니다. 은은한 단맛을 원하신다면 시그니처 아이스 라떼인 '놀라'도 훌륭한 선택입니다.
*   **매장 분위기 및 좌석 편의성:**
    *   화이트 톤의 극도로 심플하고 차분한 인테리어입니다. 명동교자의 북적이고 활기찬 분위기와 완전히 대비되는 **차분하고 정돈된 분위기** 속에서 가볍게 커피에 집중하며 대화를 나누기 좋습니다. 통창을 통해 들어오는 채광이 좋아 밝은 분위기를 선호하시는 분들께 추천합니다.

---

**요약 가이드:**
*   **입안의 마늘 냄새를 빠르게 없애고 넓은 공간에서 편하게 쉬고 싶다면?** ➡️ **맷차 명동본점**
*   **소화도 시킬 겸 걸어가서 이색적인 분위기와 깔끔한 드립커피를 즐기고 싶다면?** ➡️ **커피한약방**
*   **세련되고 조용한 공간에서 퀄리티 높은 커피 맛 자체에 집중하고 싶다면?** ➡️ **블루보틀 명동**

## 🚗🚶 5. 음식점 근처 맞춤 여행 코스
식사 후 즐길 수 있는 **차량 드라이브 코스**와 **도보 산책 코스**를 Google Maps Directions API 및 Gemini 3.5 Flash로 설계합니다.


In [7]:
# 5.1 주변 대표 관광 명소 탐색 (Places API)
tourist_url = "https://places.googleapis.com/v1/places:searchNearby"
tourist_headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": MAPS_API_KEY,
    "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.rating,places.location,places.primaryType"
}
tourist_body = {
    "includedTypes": ["tourist_attraction", "park", "historical_landmark", "museum"],
    "maxResultCount": 6,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": target_lat, "longitude": target_lng},
            "radius": 3000.0  # 반경 3km
        }
    },
    "languageCode": "ko"
}

tourist_raw = requests.post(tourist_url, headers=tourist_headers, json=tourist_body).json()
attractions = []
for t in tourist_raw.get("places", []):
    attractions.append({
        "명소명": t.get("displayName", {}).get("text"),
        "평점": f"⭐ {t.get('rating', 'N/A')}",
        "주소": t.get("formattedAddress"),
        "coords": (t["location"]["latitude"], t["location"]["longitude"])
    })

df_attractions = pd.DataFrame([{"명소명": a["명소명"], "평점": a["평점"], "주소": a["주소"]} for a in attractions])
print("🏛️ [주변 주요 관광/문화 명소 목록]")
display(df_attractions)

# 5.2 Gemini 3.5 Flash를 활용한 차량 드라이브 코스 & 도보 산책 코스 종합 생성
itinerary_prompt = f"""
출발점 (식사 장소): '{TARGET_RESTAURANT}' (주소: {details_data.get('formattedAddress')})
주변 탐색된 관광 명소 및 문화 유적:
{json.dumps([a['명소명'] for a in attractions], ensure_ascii=False)}

위 정보를 바탕으로 식사 후 이어지는 완벽한 2가지 테마 여행 코스를 작성해주세요:

---
### 🚗 1. 차량 드라이브 코스 (Half-Day Driving Course)
- **추천 대상**: 드라이브, 야경, 뷰포인트 감상을 원하는 방문객
- **추천 경로**: {TARGET_RESTAURANT} ➡️ [주요 드라이브 명소 1~2곳 (예: 남산 순환로, 북악스카이웨이, 한강 뷰포인트 등)] ➡️ [일몰/야경 카페]
- **포인트**: 각 스팟별 주차 편의성, 추천 드라이브 시간대, 예상 차량 소요시간

---
### 🚶 2. 힐링 도보 산책 코스 (Pedestrian Walking Tour)
- **추천 대상**: 식사 후 가볍게 소화시키며 도심 문화를 즐기려는 도보 여행자
- **추천 경로**: {TARGET_RESTAURANT} ➡️ [도보 5~15분 거리 명소 (예: 명동성당, 청계천 산책로, 남산골 한옥마을 등)]
- **포인트**: 총 보행 시간 (약 20~40분), 포토존, 산책 힐링 포인트
"""

print("🗺️ Gemini 3.5 Flash가 맞춤형 차량/도보 여행 코스를 설계 중입니다...\n")
response_itinerary = ai_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=itinerary_prompt
)
display(Markdown(response_itinerary.text))


🏛️ [주변 주요 관광/문화 명소 목록]


,명소명,평점,주소
0,경복궁,⭐ 4.6,대한민국 서울특별시 종로구 사직로 161
1,광장시장,⭐ 4.2,대한민국 서울특별시 종로구 청계천로 88
2,N서울타워,⭐ 4.5,대한민국 서울특별시 용산구 남산공원길 105
3,북촌 한옥마을,⭐ 4.4,대한민국 서울특별시 종로구 계동길
4,명동거리,⭐ 4.4,대한민국 서울특별시 중구 명동2가
5,남대문시장,⭐ 4.2,대한민국 서울특별시 중구 남대문시장4길 21


🗺️ Gemini 3.5 Flash가 맞춤형 차량/도보 여행 코스를 설계 중입니다...



명동교자 본점(퇴계로 인근)에서 든든하고 만족스러운 식사를 마친 후, 서울의 매력을 200% 즐길 수 있는 **차량 드라이브 코스**와 **도보 산책 코스** 두 가지를 제안해 드립니다. 취향에 맞는 완벽한 여정을 선택해 보세요!

---

### 🚗 1. 차량 드라이브 코스 (Half-Day Driving Course)
> **추천 대상**: 식사 후 편안하게 차 안에서 서울의 스카이라인을 감상하고, 아름다운 낙조와 야경을 만끽하고 싶은 분들

```
[명동교자] ➡️ (차량 10분) ➡️ [남산 소월로 드라이브] ➡️ (차량 25분) ➡️ [북악스카이웨이 팔각정] ➡️ (차량 5분) ➡️ [평창동/부암동 뷰포인트 카페]
```

#### 📍 스팟별 상세 정보

1. **남산 소월로 드라이브 코스 (명동 ➡️ 숭례문 ➡️ 한남동 방향)**
   * **설명**: 명동교자에서 나와 남산의 순환도로인 '소월로'로 진입합니다. 흐드러진 가로수 사이로 보이는 N서울타워와 서울 시내 전경이 일품인 대표 드라이브 코스입니다.
   * **드라이브 팁**: 해 질 무렵(일몰 30분 전)에 달리면 붉게 물드는 서울 도심을 감상할 수 있습니다.

2. **북악스카이웨이 팔각정**
   * **설명**: 서울 드라이브의 성지이자 최고의 야경 명소입니다. 해발 342m에 위치해 앞쪽으로는 경복궁과 서울 도심이, 뒤쪽으로는 북한산의 웅장한 능선이 한눈에 들어옵니다.
   * **주차 편의성**: 팔각정 지하 주차장 이용 가능 (무인 정산기 운영, 주말 저녁 시간대에는 대기 줄이 길 수 있으니 평일 저녁이나 주말 늦은 밤 시간대 방문을 추천합니다).

3. **일몰/야경 추천 카페: 부암동 '산모퉁이' 또는 평창동 '더피아노'**
   * **설명**: 북악스카이웨이 하산 길에 위치한 뷰 맛집 카페들입니다. 특히 드라마 '커피프린스 1호점' 촬영지인 '산모퉁이' 야외 테라스에서는 성곽길과 서울 야경을 품에 안고 따뜻한 차 한 잔의 여유를 즐길 수 있습니다.

* **추천 드라이브 시간대**: 17:30 ~ 21:00 (노을부터 야경까지 감상 가능한 시간)
* **예상 차량 소요 시간**: 약 40분 ~ 1시간 (드라이브 및 이동 시간 기준, 트래픽 제외)

---

### 🚶 2. 힐링 도보 산책 코스 (Pedestrian Walking Tour)
> **추천 대상**: 마늘 향 가득한 칼국수와 만두를 맛있게 먹은 후, 가볍게 소화를 시키며 서울의 역사와 이국적인 정취를 동시에 느끼고 싶은 도보 여행자

```
[명동교자 본점] ➡️ (도보 5분) ➡️ [명동성당] ➡️ (도보 10분) ➡️ [남산예장공원 & 남산 오르미] ➡️ (도보 10분) ➡️ [명동 재미로 & 만화거리]
```

#### 📍 스팟별 상세 정보

1. **명동성당 (도보 5분 / 약 300m)**
   * **힐링 포인트**: 번화한 명동거리 한복판에 우뚝 솟은 고딕 양식의 붉은 벽돌 성당입니다. 성당 뒤편의 성모 동산과 고요한 산책로는 명동에서 가장 조용하고 평화로운 공간입니다. 지하의 복합문화공간 '1898 광장'에서 서점이나 소품샵을 둘러보기 좋습니다.
   * **포토존**: 성당 진입 계단 아래에서 성당 전체가 나오도록 아래에서 위로 찍는 구도를 추천합니다.

2. **남산예장공원 & 남산 오르미 (도보 10분 / 약 650m)**
   * **힐링 포인트**: 과거 중앙정보부 터를 시민들의 휴식 공간으로 재탄생시킨 곳입니다. 탁 트인 녹지와 소나무 숲길이 조성되어 있어 걷기 좋습니다. 공원 끝자락에 있는 무료 경사형 엘리베이터 '남산 오르미'를 타면 남산 케이블카 승강장까지 편하게 이동하며 명동 일대를 조망할 수 있습니다.
   * **포토존**: 남산예장공원 내 붉은색 지붕의 우체통 미술관 앞.

3. **명동 재미로 (만화거리) (도보 10분 / 약 600m)**
   * **힐링 포인트**: 명동역 3번 출구에서 남산동 공영주차장까지 이어지는 언덕길로, 골목 구석구석 귀여운 국산 캐릭터(타요, 라바 등) 벽화와 조형물이 설치되어 있어 아기자기한 재미를 줍니다. 가벼운 발걸음으로 명동역으로 되돌아오기 좋은 코스입니다.

* **총 보행 시간**: 약 25분 ~ 30분 (순수 걷는 시간 기준, 약 1.5km)
* **소요 시간**: 사진 촬영 및 휴식 포함 약 1시간 ~ 1시간 30분

## 📊 6. 최종 종합 요약 및 활용 가이드

본 노트북은 **Google Maps Platform API**의 실시간 지리정보(장소 상세, 편의시설, 리뷰, 주변 검색)를 데이터 파이프라인으로 삼고, **Gemini 3.5 Flash**의 다국어 텍스트 이해 및 추론 역량을 결합하여 상용 서비스 수준의 **AI 맛집 & 여행 플래너**를 성공적으로 구축하였습니다.

---

### 💡 다른 지역 / 음식점으로 확장하는 방법
노트북 상단의 `TARGET_RESTAURANT` 변수를 원하는 음식점으로 변경하고 전체 셀을 다시 실행하면 모든 분석과 여행 코스가 즉시 새로 생성됩니다:
```python
TARGET_RESTAURANT = "강남파이낸스센터 인근 맛집"  # 또는 "성수동 소문난 성수 감자탕", "해운대 암소갈비집" 등
```
